## Interaktive Karte zu Hypothese 2: Fläche fürs Geld vs. Nähe zum Zentrum (v3)

**Kunde:** Thomas Hansen, 5 Kinder, wenig Geld, sucht ein **großes Haus**. Er fragt: *Wo bekomme ich am meisten Fläche für mein Geld?*
**Hypothese 2:** Je weiter ein großes Haus (≥ 4 Schlafzimmer) von Downtown Seattle entfernt ist, desto niedriger ist sein Preis pro qm. **Bestätigt** (Spearman ρ ≈ −0,61, etwa −2 % pro km).

### Warum andere Regler als in v1?
Der Familienindex in v1 (Grundstücksgröße `qm_lot15` + Familienanteil) beantwortet eher die Frage „Wo wohnen Familien?“. H2 geht es aber um **Fläche, Preis und Lage**. Prüfung auf den großen Häusern (≥ 4 SZ, Spearman ρ):

| Zusammenhang | ρ | Bedeutung |
|---|---|---|
| Entfernung ↔ Preis pro qm (alle Häuser) | −0,61 | Kern von H2 |
| Entfernung ↔ Preis pro qm (ZIP-Ebene) | −0,71 | Effekt ist auch je ZIP klar sichtbar |
| **innerhalb 500–700 Tsd. $:** Entfernung ↔ Wohnfläche `qm_living` | **+0,48** | Bei festem Budget gibt es weiter draußen mehr m² |
| Preis pro qm ↔ Preis (ZIP-Ebene) | 0,93 | qm-Preis je ZIP markiert fast nur „teure ZIPs“, also kein eigener Regler |
| Entfernung ↔ `qm_lot` (ZIP-Ebene) | 0,56 | Grundstück ist v. a. ein Lage-Effekt, also nicht doppelt zählen |

**Entscheidung:**
* **Preisspanne (von–bis)** bleibt. Sie ist Thomas' harte Grenze.
* **Neu: Schlafzimmer 4+ / 5+ / 6+.** 4+ ist die H2-Definition von „groß“. Bei 5 Kindern sind 5+ (1.934 Verkäufe) oder 6+ (333) realistisch.
* **Gewichtungsregler neu: Wohnfläche ↔ Nähe zu Downtown.** Das ist genau der Zielkonflikt aus H2: mehr m² gegen kürzere Wege.
  * **Wohnfläche:** Median-`qm_living` der passenden Häuser **in der gewählten Spanne**. Das beantwortet direkt: *Wie viel Haus bekomme ich hier für mein Geld?*
  * **Nähe:** Median-Luftlinie der Verkäufe im ZIP zu Downtown (47,6062 N / 122,3321 W), invertiert und fest.
* **Unverändert:** Säulenhöhe = Anzahl Häuser in der Spanne, Farbe = Index (Braun → Petrol), Top-5-Empfehlung, Seattle-Grenze. Neu ist ein roter Punkt für Downtown als Bezugspunkt.
* **Tooltip:** Median-m², Median-$/m² in der Spanne, km zu Downtown, dazu der qm-Preis des ganzen ZIPs (H2-Kennzahl) als Info.

### Neu in v3
* **Feste Vergleichsbasis für die Flächenkomponente.** In v2 wurde gegen die ZIPs mit mindestens 3 Treffern in der aktuellen Auswahl verglichen. Die Basis wanderte also mit Spanne und Schlafzimmerzahl, und Farben waren zwischen Einstellungen nicht vergleichbar. Jetzt wird einmal in Python eine Referenz gebildet: **alle großen Häuser (≥ 4 SZ, je Haus der letzte Verkauf) im gesamten Datensatz**, ohne Filter. Sie wird als `reference_living` übergeben. Die Nähekomponente war schon fest.
* `MIN_HOUSES_FOR_RANKING` gilt nur noch für die Top-5-Liste und den Hinweis „wenige Treffer“ im Tooltip.
* **Schalter `REFERENCE_METRIC`:** `"living"` (Standard, mehr m² = besser) oder `"price_per_qm"` (Median-Preis pro m² in der Spanne, niedriger = besser, Referenz = `preis_pro_qm` aller großen Häuser).
* **Stadtgrenze und Downtown-Punkt** lassen sich in der Seitenleiste per Checkbox ein- und ausblenden (Startzustand: `SHOW_BOUNDARY_START`, `SHOW_DOWNTOWN_START`).
* **Folge der festen Basis:** Die Index-Farben sagen jetzt absolut etwas aus. Bei einer teuren Spanne liegen viele ZIPs weit oben auf der Flächenachse, bei einer billigen weit unten. Die Rangfolge in der Top-5-Liste bleibt ein Vergleich innerhalb der Spanne.

**Einschränkung:** Luftlinie statt Fahrzeit. Schulen, Sicherheit und Parks sind nicht im Datensatz. Verkäufe 2014/15.
Die Zelle ist eigenständig ausführbar und öffnet die Karte automatisch in **Chrome** (`OPEN_IN_CHROME = False` schaltet das ab).


In [ ]:
# %%
# ============================================================
# Preisspanne (von–bis) – Variante 3, ausgerichtet auf Hypothese 2
# Feste Vergleichsbasis für die Flächenkomponente,
# Stadtgrenze und Downtown-Punkt per Schalter.
# Eigenständig ausführbar, öffnet die Karte automatisch in Chrome.
# ============================================================

import copy
import json
import math
import os
import subprocess
import sys
import urllib.request
import webbrowser

from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from threading import Thread
from urllib.parse import quote, urlencode

import pandas as pd

from IPython.display import IFrame, display


# ============================================================
# 1. Einstellungen
# ============================================================

OUTPUT_HTML_RANGE = "seattle_preisspanne_interaktiv_claude_v3.html"

# Karte nach dem Erzeugen automatisch in Chrome öffnen
OPEN_IN_CHROME = True

ZIP_GEOJSON_URL = (
    "https://raw.githubusercontent.com/OpenDataDE/"
    "State-zip-code-GeoJSON/master/"
    "wa_washington_zip_codes_geo.min.json"
)

SEATTLE_BOUNDARY_SERVICE = (
    "https://data.wsdot.wa.gov/arcgis/rest/services/"
    "Shared/PoliAdminBndryData/FeatureServer/1/query"
)

# Stadtzentrum wie in Hypothese 2: Downtown Seattle
DOWNTOWN_LAT = 47.6062
DOWNTOWN_LON = -122.3321

# Preisspanne: Regler in 100.000-$-Schritten, Anzeige als 1,0 = 100.000 $
PRICE_UNIT = 100_000
RANGE_MIN = 100_000
RANGE_MAX = 2_000_000
RANGE_START_LOW = 500_000
RANGE_START_HIGH = 700_000

# King County Daten: price ist in US-Dollar.
CURRENCY = "$"

# Schlafzimmer-Auswahl: großes Haus = mindestens 4 (Definition H2).
# Thomas hat 5 Kinder, daher auch 5+ und 6+ wählbar.
BEDROOM_OPTIONS = [4, 5, 6]
BEDROOMS_START = 4

# Startgewichtung:
# Wohnfläche in der Spanne und Nähe zum Zentrum ergeben zusammen 100 %.
WEIGHT_SPACE_START = 0.50
WEIGHT_STEP_PERCENT = 5

# Kennzahl der Flächenkomponente:
# "living"       -> Median-Wohnfläche in der Spanne (mehr = besser)
# "price_per_qm" -> Median-Preis pro m² in der Spanne (weniger = besser)
# Verglichen wird immer mit einer festen Referenz: alle großen Häuser
# (ab min(BEDROOM_OPTIONS) Schlafzimmern) im gesamten Datensatz.
REFERENCE_METRIC = "living"

# Visuelle Höhe pro passendem Haus
HEIGHT_PER_HOUSE = 20

# Startansicht
CENTER_LAT = 47.56
CENTER_LON = -122.12
ZOOM = 9.65
PITCH = 50
BEARING = 0

# Farbgrenzen des Index
INDEX_RED = 0
INDEX_YELLOW = 50
INDEX_GREEN = 100

# Farben (farbenblind-tauglich: Braun -> Creme -> Petrol)
COLOR_RED = [166, 97, 26]
COLOR_YELLOW = [246, 232, 195]
COLOR_GREEN = [1, 133, 113]
COLOR_NO_MATCH = [190, 190, 190]

# Seattle-Stadtgrenze
BOUNDARY_COLOR = [0, 75, 255]
BOUNDARY_WIDTH = 4

# Markierung Downtown
DOWNTOWN_COLOR = [200, 30, 30]

# Startzustand der Schalter für Stadtgrenze und Downtown-Punkt
SHOW_BOUNDARY_START = True
SHOW_DOWNTOWN_START = True

# Zahlen direkt auf den Flächen
LABEL_SIZE = 17
LABEL_COLOR = [15, 15, 15, 255]

# False: Schrift liegt parallel zur Fläche.
# True: Schrift bleibt zur Kamera ausgerichtet.
LABEL_BILLBOARD = False

# Empfehlungsliste: beste ZIP-Gebiete in der Spanne.
# Unter MIN_HOUSES_FOR_RANKING Treffern ist der Median
# zu unsicher – solche Gebiete kommen nicht in die Liste
# und bekommen im Tooltip einen Hinweis.
# Die Vergleichsbasis des Index hängt davon nicht ab.
TOP_N = 5
MIN_HOUSES_FOR_RANKING = 3


# ============================================================
# 2. Einstellungen prüfen
# ============================================================

if not 0 <= WEIGHT_SPACE_START <= 1:
    raise ValueError(
        "WEIGHT_SPACE_START muss zwischen 0 und 1 liegen."
    )

if (
    WEIGHT_STEP_PERCENT <= 0
    or 100 % WEIGHT_STEP_PERCENT != 0
):
    raise ValueError(
        "WEIGHT_STEP_PERCENT muss positiv sein "
        "und 100 ohne Rest teilen."
    )

if not INDEX_RED < INDEX_YELLOW < INDEX_GREEN:
    raise ValueError(
        "INDEX_RED < INDEX_YELLOW < INDEX_GREEN muss gelten."
    )

if not (
    RANGE_MIN <= RANGE_START_LOW
    < RANGE_START_HIGH <= RANGE_MAX
):
    raise ValueError(
        "RANGE_MIN <= RANGE_START_LOW < "
        "RANGE_START_HIGH <= RANGE_MAX muss gelten."
    )

if PRICE_UNIT <= 0:
    raise ValueError(
        "PRICE_UNIT muss größer als 0 sein."
    )

if REFERENCE_METRIC not in ("living", "price_per_qm"):
    raise ValueError(
        'REFERENCE_METRIC muss "living" oder "price_per_qm" sein.'
    )

if BEDROOMS_START not in BEDROOM_OPTIONS:
    raise ValueError(
        "BEDROOMS_START muss in BEDROOM_OPTIONS enthalten sein."
    )

if HEIGHT_PER_HOUSE < 0:
    raise ValueError(
        "HEIGHT_PER_HOUSE darf nicht negativ sein."
    )


# ============================================================
# 3. Vorherigen Server dieser Zelle schließen
# ============================================================

if "_range_server_v3" in globals():
    _range_server_v3.shutdown()
    _range_server_v3.server_close()


# ============================================================
# 4. JSON herunterladen
# ============================================================

def load_json(url):
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0",
        },
    )

    with urllib.request.urlopen(
        request,
        timeout=90,
    ) as response:
        result = json.load(response)

    if isinstance(result, dict) and "error" in result:
        raise RuntimeError(
            f"Fehler beim Laden der Geodaten: {result['error']}"
        )

    return result


# ============================================================
# 5. CSV laden
# ============================================================

possible_paths = [
    "eda_tk_cleaned.csv",
    "data/eda_tk_cleaned.csv",
    "./eda_tk_cleaned.csv",
]

file_path = next(
    (
        path
        for path in possible_paths
        if os.path.exists(path)
    ),
    None,
)

if file_path is None:
    raise FileNotFoundError(
        "Die Datei 'eda_tk_cleaned.csv' wurde nicht gefunden."
    )

df = pd.read_csv(
    file_path,
    dtype={
        "house_id": "string",
        "id": "string",
        "zipcode": "string",
    },
)

# Deine bereinigte CSV verwendet house_id.
if "house_id" in df.columns:
    df["id"] = df["house_id"]

required_columns = [
    "id",
    "date",
    "zipcode",
    "bedrooms",
    "price",
    "qm_living",
    "lat",
    "long",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Diese Spalten fehlen: {missing_columns}"
    )


# ============================================================
# 6. Daten bereinigen
# ============================================================

df["zipcode"] = (
    df["zipcode"]
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

df["id"] = (
    df["id"]
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

df.loc[df["id"].eq(""), "id"] = pd.NA

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce",
)

numeric_columns = [
    "bedrooms",
    "price",
    "qm_living",
    "lat",
    "long",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce",
    )

df[numeric_columns] = df[numeric_columns].replace(
    [float("inf"), float("-inf")],
    float("nan"),
)

df = df.loc[
    df["zipcode"].str.fullmatch(
        r"\d{5}",
        na=False,
    )
].copy()

for column in [
    "price",
    "qm_living",
]:
    df.loc[df[column] <= 0, column] = float("nan")

df.loc[
    ~df["lat"].between(-90, 90)
    | ~df["long"].between(-180, 180),
    ["lat", "long"],
] = float("nan")


# ============================================================
# 7. Entfernung zum Zentrum (wie in Hypothese 2)
# ============================================================

def distance_km(lat, lon):
    # Luftlinie (Haversine) zu Downtown Seattle
    lat1 = math.radians(DOWNTOWN_LAT)
    lon1 = math.radians(DOWNTOWN_LON)

    lat2 = lat.map(math.radians)
    lon2 = lon.map(math.radians)

    a = (
        ((lat2 - lat1) / 2).map(math.sin) ** 2
        + math.cos(lat1)
        * lat2.map(math.cos)
        * ((lon2 - lon1) / 2).map(math.sin) ** 2
    )

    return 2 * 6371 * a.map(
        lambda value: math.asin(math.sqrt(value))
        if pd.notna(value) else float("nan")
    )


df["entfernung_km"] = distance_km(
    df["lat"],
    df["long"],
)

df["preis_pro_qm"] = df["price"] / df["qm_living"]


# ============================================================
# 8. ZIP-Geometrien und Seattle-Stadtgrenze laden
# ============================================================

zip_geojson = load_json(ZIP_GEOJSON_URL)

boundary_query = urlencode({
    "where": "CityName = 'Seattle'",
    "outFields": "CityName",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson",
})

seattle_boundary = load_json(
    f"{SEATTLE_BOUNDARY_SERVICE}?{boundary_query}"
)

if not seattle_boundary.get("features"):
    raise ValueError(
        "Der Grenzdienst hat keine "
        "Seattle-Stadtgrenze geliefert."
    )

geometry_zipcodes = {
    str(
        feature.get("properties", {}).get(
            "ZCTA5CE10",
            "",
        )
    ).zfill(5)
    for feature in zip_geojson["features"]
    if feature.get("geometry") is not None
}

df = df.loc[
    df["zipcode"].isin(geometry_zipcodes)
].copy()

if df.empty:
    raise ValueError(
        "Keine Beobachtungen mit passenden ZIP-Geometrien."
    )


# ============================================================
# 9. Große Häuser: ein Eintrag pro Haus (letzter Verkauf)
# ============================================================

min_bedrooms = min(BEDROOM_OPTIONS)

df_large = df.loc[
    (df["bedrooms"] >= min_bedrooms)
    & df["price"].notna()
    & df["qm_living"].notna()
    & df["id"].notna()
].copy()

if df_large.empty:
    raise ValueError(
        "Keine großen Häuser mit gültigem Preis, "
        "gültiger Wohnfläche und Haus-ID."
    )

# Letzter Verkauf je Haus = aktuellster Marktwert
df_houses = (
    df_large
    .sort_values("date")
    .groupby(
        ["zipcode", "id"],
        as_index=False,
    )
    .last()
    .sort_values(["zipcode", "price"])
)


# Feste Vergleichsbasis für die Flächenkomponente:
# alle großen Häuser (ab min(BEDROOM_OPTIONS) Schlafzimmern),
# je Haus der letzte Verkauf, ohne Filter auf Preisspanne
# oder aktuelle Zimmerwahl. Gleiche Farbe = gleicher Wert,
# egal welche Einstellung gerade aktiv ist.
reference_living = sorted(
    round(float(value), 1)
    for value in df_houses["qm_living"]
)

reference_price_per_qm = sorted(
    round(float(value), 1)
    for value in df_houses["preis_pro_qm"]
)


# ============================================================
# 10. Lage je ZIP: Entfernung zum Zentrum
# ============================================================

# Median-Entfernung aller Verkäufe im ZIP = typische Lage.
df_zip = (
    df.dropna(subset=["entfernung_km"])
    .groupby("zipcode", as_index=False)
    .agg(
        median_km=("entfernung_km", "median"),
        anzahl_umfeld=("zipcode", "size"),
    )
)

# qm-Preis großer Häuser im ganzen ZIP (Kennzahl aus H2, nur Info)
ppq_by_zip = (
    df_large
    .groupby("zipcode")["preis_pro_qm"]
    .median()
)

df_zip = df_zip.loc[
    df_zip["zipcode"].isin(df_houses["zipcode"])
].copy()

if df_zip.empty:
    raise ValueError(
        "Keine ZIP-Gebiete mit großen Häusern."
    )


def percentile_rank_01(series):
    if len(series) <= 1:
        return pd.Series(
            0.5,
            index=series.index,
            dtype=float,
        )

    return (
        (series.rank(method="average") - 1)
        / (len(series) - 1)
    )


# Nähe-Komponente: 1 = nächstes ZIP, 0 = am weitesten weg.
# Fest über alle ZIP-Gebiete, unabhängig von den Reglern.
df_zip["near_percentile"] = 1 - percentile_rank_01(
    df_zip["median_km"]
)

zip_by_code = (
    df_zip
    .set_index("zipcode")
    .to_dict("index")
)


# ============================================================
# 11. Positionen für die Zahlen
# ============================================================

df_positions = (
    df[["zipcode", "lat", "long"]]
    .dropna(subset=["lat", "long"])
    .copy()
)

label_positions = {}

for zipcode, group in df_positions.groupby("zipcode"):
    center_lat = group["lat"].median()
    center_lon = group["long"].median()

    # Tatsächlicher Hausstandort nahe der Mitte.
    distance = (
        (group["lat"] - center_lat) ** 2
        + (
            (group["long"] - center_lon) * 0.67
        ) ** 2
    )

    nearest = group.loc[distance.idxmin()]

    label_positions[str(zipcode)] = [
        float(nearest["long"]),
        float(nearest["lat"]),
    ]


# ============================================================
# 12. Daten für die HTML-Karte vorbereiten
# ============================================================

houses_by_zip = {
    str(zipcode): {
        # Nach Preis sortiert -> Spanne per Binärsuche
        "prices": [float(v) for v in group["price"]],
        "living": [round(float(v), 1) for v in group["qm_living"]],
        "beds": [int(v) for v in group["bedrooms"]],
    }
    for zipcode, group in df_houses.groupby("zipcode")
}

features = []
zip_stats = {}

for feature in zip_geojson["features"]:
    zipcode = str(
        feature.get("properties", {}).get(
            "ZCTA5CE10",
            "",
        )
    ).zfill(5)

    if (
        zipcode not in zip_by_code
        or feature.get("geometry") is None
    ):
        continue

    stats = zip_by_code[zipcode]

    new_feature = copy.deepcopy(feature)
    new_feature["properties"] = {
        "zipcode": zipcode,
    }

    features.append(new_feature)

    zip_stats[zipcode] = {
        "nearPercentile": float(
            stats["near_percentile"]
        ),
        "medianKm": float(
            stats["median_km"]
        ),
        "zipPricePerQm": float(
            ppq_by_zip[zipcode]
        ),
        "observations": int(
            stats["anzahl_umfeld"]
        ),
        "houses": houses_by_zip[zipcode],
        "labelPosition": label_positions.get(
            zipcode
        ),
    }

if not features:
    raise ValueError(
        "Keine darstellbaren ZIP-Flächen."
    )

payload_range = {
    "reference_living": reference_living,
    "reference_price_per_qm": reference_price_per_qm,
    "geojson": {
        "type": "FeatureCollection",
        "features": features,
    },
    "boundary": seattle_boundary,
    "stats": zip_stats,
    "config": {
        "budgetMin": RANGE_MIN,
        "budgetMax": RANGE_MAX,
        "budgetStep": PRICE_UNIT,
        "rangeStartLow": RANGE_START_LOW,
        "rangeStartHigh": RANGE_START_HIGH,
        "priceUnit": PRICE_UNIT,
        "currency": CURRENCY,
        "bedroomOptions": BEDROOM_OPTIONS,
        "bedroomsStart": BEDROOMS_START,
        "weightSpaceStart": WEIGHT_SPACE_START,
        "weightStepPercent": WEIGHT_STEP_PERCENT,
        "referenceMetric": REFERENCE_METRIC,
        "referenceHouses": len(reference_living),
        "referenceMinBeds": min_bedrooms,
        "heightPerHouse": HEIGHT_PER_HOUSE,
        "topN": TOP_N,
        "minHousesForRanking": MIN_HOUSES_FOR_RANKING,
        "indexRed": INDEX_RED,
        "indexYellow": INDEX_YELLOW,
        "indexGreen": INDEX_GREEN,
        "red": COLOR_RED,
        "yellow": COLOR_YELLOW,
        "green": COLOR_GREEN,
        "noMatch": COLOR_NO_MATCH,
        "boundaryColor": BOUNDARY_COLOR,
        "boundaryWidth": BOUNDARY_WIDTH,
        "downtown": [DOWNTOWN_LON, DOWNTOWN_LAT],
        "downtownColor": DOWNTOWN_COLOR,
        "showBoundaryStart": SHOW_BOUNDARY_START,
        "showDowntownStart": SHOW_DOWNTOWN_START,
        "labelSize": LABEL_SIZE,
        "labelColor": LABEL_COLOR,
        "labelBillboard": LABEL_BILLBOARD,
        "view": {
            "latitude": CENTER_LAT,
            "longitude": CENTER_LON,
            "zoom": ZOOM,
            "pitch": PITCH,
            "bearing": BEARING,
        },
    },
}


# ============================================================
# 13. HTML mit Preisspanne, Schlafzimmern und Gewichtung
# ============================================================

html_template_range = r"""
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="utf-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1"
    >

    <title>Seattle Fläche und Lage</title>

    <script src="https://unpkg.com/deck.gl@8.9.36/dist.min.js"></script>

    <script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>

    <link
        href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css"
        rel="stylesheet"
    >

    <style>
        * {
            box-sizing: border-box;
        }

        html,
        body {
            margin: 0;
            width: 100%;
            height: 100%;
            overflow: hidden;
            font-family: Arial, sans-serif;
            color: #222;
        }

        #layout {
            display: flex;
            width: 100%;
            height: 100%;
        }

        #sidebar {
            width: 280px;
            min-width: 280px;
            padding: 20px 18px;
            background: #fff;
            border-right: 1px solid #ddd;
            overflow-y: auto;
            z-index: 2;
        }

        #map {
            position: relative;
            flex: 1;
            min-width: 0;
            height: 100%;
        }

        h2 {
            margin: 0 0 20px;
            font-size: 19px;
            line-height: 1.35;
        }

        .section {
            margin-top: 18px;
            padding-top: 15px;
            border-top: 1px solid #ddd;
        }

        .label {
            font-size: 13px;
            font-weight: 700;
        }

        #budgetValue {
            margin: 9px 0 12px;
            font-size: 25px;
            font-weight: 700;
        }

        input[type="range"] {
            width: 100%;
            cursor: pointer;
            accent-color: #2459aa;
        }

        .dual {
            position: relative;
            height: 28px;
            margin-top: 4px;
        }

        .dual-track,
        .dual-fill {
            position: absolute;
            top: 12px;
            height: 4px;
            border-radius: 2px;
        }

        .dual-track {
            left: 0;
            right: 0;
            background: #ddd;
        }

        .dual-fill {
            background: #2459aa;
        }

        .dual input[type="range"] {
            position: absolute;
            left: 0;
            top: 0;
            width: 100%;
            height: 28px;
            margin: 0;
            background: none;
            pointer-events: none;
            -webkit-appearance: none;
            appearance: none;
        }

        .dual input[type="range"]::-webkit-slider-runnable-track {
            background: none;
        }

        .dual input[type="range"]::-moz-range-track {
            background: none;
        }

        .dual input[type="range"]::-webkit-slider-thumb {
            pointer-events: auto;
            -webkit-appearance: none;
            width: 18px;
            height: 18px;
            border-radius: 50%;
            background: #2459aa;
            border: 2px solid #fff;
            box-shadow: 0 0 0 1px #2459aa;
            cursor: pointer;
        }

        .dual input[type="range"]::-moz-range-thumb {
            pointer-events: auto;
            width: 14px;
            height: 14px;
            border-radius: 50%;
            background: #2459aa;
            border: 2px solid #fff;
            box-shadow: 0 0 0 1px #2459aa;
            cursor: pointer;
        }

        #weightSlider {
            margin-top: 12px;
        }

        .range,
        .scale {
            display: flex;
            justify-content: space-between;
            gap: 8px;
            margin-top: 5px;
            font-size: 11px;
        }

        .small {
            font-size: 12px;
            line-height: 1.55;
        }

        .muted {
            color: #666;
            font-size: 11px;
            line-height: 1.5;
        }

        #gradient {
            height: 13px;
            margin-top: 10px;
            border-radius: 4px;
        }

        #weights {
            margin-top: 9px;
            line-height: 1.7;
        }

        .boundary-key {
            display: inline-block;
            width: 28px;
            height: 0;
            margin-right: 7px;
            border-top: 4px solid;
            vertical-align: middle;
        }

        #error {
            display: none;
            margin-top: 15px;
            color: #a00000;
            font-size: 12px;
            white-space: pre-wrap;
        }

        #topList {
            margin: 8px 0 0;
            padding-left: 20px;
        }

        #topList li {
            margin-bottom: 6px;
        }

        .swatch {
            display: inline-block;
            width: 10px;
            height: 10px;
            margin-right: 5px;
            border-radius: 2px;
            vertical-align: middle;
        }

        button {
            padding: 8px 10px;
            border: 1px solid #bbb;
            border-radius: 5px;
            background: #f7f7f7;
            cursor: pointer;
        }

        .seg {
            display: flex;
            gap: 6px;
            margin-top: 8px;
        }

        .seg button {
            flex: 1;
            font-weight: 700;
        }

        .seg button.active {
            background: #2459aa;
            border-color: #2459aa;
            color: #fff;
        }

        .dot-key {
            display: inline-block;
            width: 12px;
            height: 12px;
            margin: 8px 7px 0 8px;
            border: 2px solid #fff;
            border-radius: 50%;
            box-shadow: 0 0 0 1px #999;
            vertical-align: middle;
        }

        code {
            font-size: 10px;
        }

        .toggle {
            display: flex;
            align-items: center;
            margin-top: 8px;
            cursor: pointer;
        }

        .toggle input {
            margin: 0 8px 0 0;
            accent-color: #2459aa;
            cursor: pointer;
        }

        .toggle .boundary-key,
        .toggle .dot-key {
            flex: none;
            margin: 0 7px 0 0;
        }
    </style>
</head>

<body>
<div id="layout">

    <aside id="sidebar">

        <h2>Seattle: Wo bekomme ich in meiner Preisspanne viel Haus – und wie weit ist es in die Stadt?</h2>

        <div class="label">Preisspanne (in 100.000 $)</div>

        <div id="budgetValue"></div>

        <div class="dual">
            <div class="dual-track"></div>
            <div id="dualFill" class="dual-fill"></div>

            <input
                id="budgetLow"
                type="range"
                aria-label="Mindestpreis"
            >

            <input
                id="budgetHigh"
                type="range"
                aria-label="Höchstpreis"
            >
        </div>

        <div class="range">
            <span id="budgetMin"></span>
            <span id="budgetMax"></span>
        </div>

        <div
            class="label"
            style="margin-top: 16px;"
        >
            Mindestens Schlafzimmer
        </div>

        <div
            id="bedButtons"
            class="seg"
            role="group"
            aria-label="Mindestanzahl Schlafzimmer"
        ></div>

        <div class="section small">
            <div class="label">
                Passende Häuser
            </div>

            <div
                id="summary"
                style="margin-top: 7px;"
            ></div>
        </div>

        <div class="section small">
            <div class="label">
                Empfehlung: beste ZIP-Gebiete in der Spanne
            </div>

            <ol id="topList"></ol>

            <div class="muted">
                Sortiert nach Index.
                Nur Gebiete mit mindestens
                <span id="minHouses"></span> passenden Häusern.
            </div>
        </div>

        <div class="section small">
            <div class="label">
                Was ist dir wichtiger?
            </div>

            <div id="weights"></div>

            <input
                id="weightSlider"
                type="range"
                aria-label="Gewicht der Wohnfläche"
            >

            <div class="range">
                <span>Nur kurze Wege</span>
                <span id="spaceSliderLabel">Nur Wohnfläche</span>
            </div>

            <div
                class="muted"
                style="margin-top: 9px;"
            >
                Links: Nähe zu Downtown zählt mehr
                (Schule, Arbeit, Wege für die Kinder).<br>
                Rechts: <span id="spaceSliderHint">Möglichst
                viele m² für dein Geld</span>.<br>
                Hypothese 2: Weiter draußen gibt es mehr m²
                fürs Geld – der Regler macht diesen
                Zielkonflikt sichtbar.
            </div>
        </div>

        <div class="section small">
            <div class="label">
                Farbe: Index
            </div>

            <div id="gradient"></div>

            <div class="scale">
                <span id="indexMin"></span>
                <span id="indexMiddle"></span>
                <span id="indexMax"></span>
            </div>

            <div
                class="muted"
                style="margin-top: 9px;"
            >
                Petrol = passt gut nach deiner Gewichtung.<br>
                Braun = passt schlecht nach deiner Gewichtung.<br>
                Grau und flach = keine Treffer in der Spanne.<br><br>
                Feste Vergleichsbasis: <span id="referenceText"></span>
                Gleiche Farbe bedeutet daher bei jeder Spanne und
                Schlafzimmerzahl dasselbe.
            </div>
        </div>

        <div class="section small">
            <div class="label">
                Höhe und Zahlen = Auswahl
            </div>

            Anzahl unterschiedlicher Häuser
            innerhalb der Preisspanne.

            <div
                class="muted"
                style="margin-top: 6px;"
            >
                Gleicher Höhenmaßstab bei jeder Spanne.
                Die Gewichtung verändert nur Farben
                und Indexwerte, nicht die Anzahl der Häuser.
            </div>
        </div>

        <div class="section small">
            <div class="label">
                Einblenden
            </div>

            <label class="toggle">
                <input
                    id="showBoundary"
                    type="checkbox"
                >

                <span
                    id="boundaryKey"
                    class="boundary-key"
                ></span>

                Seattle-Stadtgrenze
            </label>

            <label class="toggle">
                <input
                    id="showDowntown"
                    type="checkbox"
                >

                <span
                    id="downtownKey"
                    class="dot-key"
                ></span>

                Downtown Seattle (Bezugspunkt der Entfernung)
            </label>
        </div>

        <div class="section">
            <button id="resetView">
                Ansicht zurücksetzen
            </button>
        </div>

        <div class="section muted">
            Historische Verkäufe 2014/15,
            keine aktuellen Angebote.<br><br>

            <b>Index</b> = gewichteter Mittelwert aus zwei
            Rängen (0–100):<br>
            • <b><span id="spaceInfoName">Wohnfläche</span></b>:
            <span id="spaceInfoText"></span><br>
            • <b>Nähe</b>: Median-Luftlinie der Verkäufe im
            ZIP zu Downtown, fest über alle Gebiete.<br><br>

            Fahrzeiten, Schulen und Sicherheit sind nicht
            im Datensatz.<br><br>

            Preise in 100.000 US-Dollar,
            z. B. 7,5 = 750.000 $. Ganz rechts (Maximum +)
            schließt alle teureren Häuser mit ein.
        </div>

        <div id="error"></div>

    </aside>

    <main id="map"></main>

</div>

<script>
const DATA = __PAYLOAD__;
const C = DATA.config;


// ------------------------------------------------------------
// Formatierung und Fehleranzeige
// ------------------------------------------------------------

const fmt = (value, digits = 0) =>
    Number(value).toLocaleString("de-DE", {
        minimumFractionDigits: digits,
        maximumFractionDigits: digits
    });

function showError(error) {
    const element = document.getElementById("error");

    element.style.display = "block";

    element.textContent =
        error && error.message
            ? error.message
            : String(error);
}

window.addEventListener("error", event => {
    showError(event.message);
});


// ------------------------------------------------------------
// Budgetregler vorbereiten
// ------------------------------------------------------------

const lowSlider =
    document.getElementById("budgetLow");

const highSlider =
    document.getElementById("budgetHigh");

[lowSlider, highSlider].forEach(slider => {
    slider.min = C.budgetMin;
    slider.max = C.budgetMax;
    slider.step = C.budgetStep;
});

lowSlider.value = C.rangeStartLow;
highSlider.value = C.rangeStartHigh;

// Preise in 100.000-$-Einheiten: 750.000 -> "7,5"
const fmtUnit = value =>
    fmt(value / C.priceUnit, 1);

// Rechter Anschlag = nach oben offen
const isOpenTop = high =>
    high >= C.budgetMax;

const fmtHigh = high =>
    fmtUnit(high) + (isOpenTop(high) ? "+" : "");

document.getElementById("budgetMin").textContent =
    fmtUnit(C.budgetMin);

document.getElementById("budgetMax").textContent =
    fmtHigh(C.budgetMax);

function currentRange() {
    return {
        low: Number(lowSlider.value),
        high: Number(highSlider.value)
    };
}

function updateBudgetLabel() {
    const {low, high} = currentRange();

    document.getElementById("budgetValue").textContent =
        fmtUnit(low) + " – " + fmtHigh(high);

    const span = C.budgetMax - C.budgetMin;
    const fill = document.getElementById("dualFill");

    fill.style.left =
        100 * (low - C.budgetMin) / span + "%";

    fill.style.width =
        100 * (high - low) / span + "%";
}


// ------------------------------------------------------------
// Schlafzimmer-Auswahl vorbereiten
// ------------------------------------------------------------

let currentMinBeds = C.bedroomsStart;

const bedButtons =
    document.getElementById("bedButtons");

C.bedroomOptions.forEach(beds => {
    const button = document.createElement("button");

    button.textContent = beds + "+";
    button.dataset.beds = beds;

    button.addEventListener("click", () => {
        currentMinBeds = beds;
        updateBedButtons();
        scheduleMapUpdate();
    });

    bedButtons.appendChild(button);
});

function updateBedButtons() {
    bedButtons.querySelectorAll("button").forEach(button => {
        const active =
            Number(button.dataset.beds) === currentMinBeds;

        button.classList.toggle("active", active);
        button.setAttribute("aria-pressed", active);
    });
}


// ------------------------------------------------------------
// Gewichtungsregler vorbereiten
// ------------------------------------------------------------

const weightSlider =
    document.getElementById("weightSlider");

weightSlider.min = 0;
weightSlider.max = 100;
weightSlider.step = C.weightStepPercent;

// Startwert auf die Schrittweite ausrichten.
weightSlider.value = Math.max(
    0,
    Math.min(
        100,
        Math.round(
            C.weightSpaceStart * 100 /
            C.weightStepPercent
        ) * C.weightStepPercent
    )
);

let currentWeightSpace =
    Number(weightSlider.value) / 100;


// ------------------------------------------------------------
// Kennzahl der Flächenkomponente (feste Referenz)
// ------------------------------------------------------------

const usePricePerQm =
    C.referenceMetric === "price_per_qm";

const spaceReference = usePricePerQm
    ? DATA.reference_price_per_qm
    : DATA.reference_living;

const spaceName = usePricePerQm
    ? "Preis pro m²"
    : "Wohnfläche";

document.getElementById("spaceSliderLabel").textContent =
    usePricePerQm
        ? "Nur Preis pro m²"
        : "Nur Wohnfläche";

document.getElementById("spaceSliderHint").textContent =
    usePricePerQm
        ? "Möglichst niedriger Preis pro m²"
        : "Möglichst viele m² für dein Geld";

document.getElementById("spaceInfoName").textContent =
    spaceName;

document.getElementById("spaceInfoText").textContent =
    (
        usePricePerQm
            ? "Median-Preis pro m² der passenden Häuser in "
              + "deiner Spanne (niedriger = besser)"
            : "Median-Wohnfläche (qm_living) der passenden "
              + "Häuser in deiner Spanne (mehr = besser)"
    ) +
    ", verglichen mit allen " +
    fmt(C.referenceHouses) +
    " großen Häusern (ab " + C.referenceMinBeds +
    " Schlafzimmern) im gesamten Datensatz. " +
    "Diese Vergleichsbasis ist fest.";

document.getElementById("referenceText").textContent =
    spaceName + " im Vergleich zu allen " +
    fmt(C.referenceHouses) + " großen Häusern (ab " +
    C.referenceMinBeds + " Schlafzimmern) im Datensatz.";

function updateWeightLabel() {
    document.getElementById("weights").innerHTML =
        "<b>" +
        fmt(currentWeightSpace * 100) +
        " %</b> " + spaceName + " in der Spanne<br>" +
        "<b>" +
        fmt((1 - currentWeightSpace) * 100) +
        " %</b> Nähe zu Downtown";
}

updateBudgetLabel();
updateBedButtons();
updateWeightLabel();

document.getElementById("minHouses").textContent =
    C.minHousesForRanking;


// ------------------------------------------------------------
// Farbskala und Legenden
// ------------------------------------------------------------

document.getElementById("indexMin").textContent =
    "≤ " + C.indexRed;

document.getElementById("indexMiddle").textContent =
    C.indexYellow;

document.getElementById("indexMax").textContent =
    "≥ " + C.indexGreen;

const rgb = color =>
    "rgb(" + color.join(",") + ")";

const yellowPosition =
    100 * (C.indexYellow - C.indexRed) /
    (C.indexGreen - C.indexRed);

document.getElementById("gradient").style.background =
    "linear-gradient(to right, " +
    rgb(C.red) + " 0%, " +
    rgb(C.yellow) + " " + yellowPosition + "%, " +
    rgb(C.green) + " 100%)";

document.getElementById("boundaryKey").style.borderColor =
    rgb(C.boundaryColor);

document.getElementById("downtownKey").style.background =
    rgb(C.downtownColor);

const showBoundaryBox =
    document.getElementById("showBoundary");

const showDowntownBox =
    document.getElementById("showDowntown");

showBoundaryBox.checked = C.showBoundaryStart;
showDowntownBox.checked = C.showDowntownStart;


// ------------------------------------------------------------
// Index -> Farbe
// ------------------------------------------------------------

function indexColor(value) {
    value = Math.max(
        C.indexRed,
        Math.min(C.indexGreen, value)
    );

    let start;
    let end;
    let fraction;

    if (value <= C.indexYellow) {
        start = C.red;
        end = C.yellow;

        fraction =
            (value - C.indexRed) /
            (C.indexYellow - C.indexRed);
    } else {
        start = C.yellow;
        end = C.green;

        fraction =
            (value - C.indexYellow) /
            (C.indexGreen - C.indexYellow);
    }

    return start.map((component, index) =>
        Math.round(
            component +
            (end[index] - component) * fraction
        )
    );
}


// ------------------------------------------------------------
// Binärsuche auf sortierten Werten
// ------------------------------------------------------------

// Anzahl Werte < value
function countBelow(values, value) {
    let left = 0;
    let right = values.length;

    while (left < right) {
        const middle = Math.floor(
            (left + right) / 2
        );

        if (values[middle] < value) {
            left = middle + 1;
        } else {
            right = middle;
        }
    }

    return left;
}

// Anzahl Werte <= value
function countAtMost(values, value) {
    let left = 0;
    let right = values.length;

    while (left < right) {
        const middle = Math.floor(
            (left + right) / 2
        );

        if (values[middle] <= value) {
            left = middle + 1;
        } else {
            right = middle;
        }
    }

    return left;
}

function median(values) {
    if (values.length === 0) {
        return null;
    }

    const sorted = [...values].sort((a, b) => a - b);
    const middle = Math.floor(sorted.length / 2);

    return sorted.length % 2
        ? sorted[middle]
        : (sorted[middle - 1] + sorted[middle]) / 2;
}

// Perzentilrang 0–1 wie pandas rank(method="average")
function percentileRank(sortedReference, value) {
    if (sortedReference.length <= 1) {
        return 0.5;
    }

    const below = countBelow(sortedReference, value);
    const atMost = countAtMost(sortedReference, value);

    const rank =
        ((below + atMost) / 2 - 0.5) /
        (sortedReference.length - 1);

    return Math.max(0, Math.min(1, rank));
}


// ------------------------------------------------------------
// Häuser eines ZIP-Gebiets in Spanne und Schlafzimmerzahl
// ------------------------------------------------------------

function selectHouses(houses, range, minBeds) {
    const start = countBelow(houses.prices, range.low);

    const end = isOpenTop(range.high)
        ? houses.prices.length
        : countAtMost(houses.prices, range.high);

    const living = [];
    const pricePerQm = [];

    for (let i = start; i < end; i++) {
        if (houses.beds[i] >= minBeds) {
            living.push(houses.living[i]);
            pricePerQm.push(
                houses.prices[i] / houses.living[i]
            );
        }
    }

    return {
        count: living.length,
        medianLiving: median(living),
        medianPricePerQm: median(pricePerQm)
    };
}


// ------------------------------------------------------------
// Seattle-Stadtgrenze in Linienpfade umwandeln
// ------------------------------------------------------------

function geometryToPaths(geometry) {
    if (!geometry) {
        return [];
    }

    switch (geometry.type) {
        case "Polygon":
            return geometry.coordinates;

        case "MultiPolygon":
            return geometry.coordinates.flatMap(
                polygon => polygon
            );

        case "LineString":
            return [geometry.coordinates];

        case "MultiLineString":
            return geometry.coordinates;

        case "GeometryCollection":
            return geometry.geometries.flatMap(
                geometryToPaths
            );

        default:
            return [];
    }
}

const boundaryPaths =
    DATA.boundary.features.flatMap(
        feature =>
            geometryToPaths(
                feature.geometry
            ).map(
                path => ({
                    path: path.map(
                        coordinate => [
                            coordinate[0],
                            coordinate[1],
                            0
                        ]
                    )
                })
            )
    );


// ------------------------------------------------------------
// Layer für aktuelle Spanne, Schlafzimmer und Gewichtung
// ------------------------------------------------------------

function buildLayers(range) {
    let total = 0;
    let matchingZipcodes = 0;

    const labels = [];

    // 1. Auswahl je ZIP berechnen
    const selections = {};

    DATA.geojson.features.forEach(feature => {
        const zipcode = feature.properties.zipcode;

        selections[zipcode] = selectHouses(
            DATA.stats[zipcode].houses,
            range,
            currentMinBeds
        );
    });

    // 2. Features mit Index und Farbe
    //    Flächenkomponente gegen feste Referenz (spaceReference)
    const features =
        DATA.geojson.features.map(feature => {

            const zipcode =
                feature.properties.zipcode;

            const stats =
                DATA.stats[zipcode];

            const selection =
                selections[zipcode];

            const count = selection.count;

            // living: mehr m² = besser
            // price_per_qm: niedriger Preis pro m² = besser
            const spaceComponent =
                count > 0
                    ? (
                        usePricePerQm
                            ? 1 - percentileRank(
                                spaceReference,
                                selection.medianPricePerQm
                            )
                            : percentileRank(
                                spaceReference,
                                selection.medianLiving
                            )
                    )
                    : null;

            const nearComponent =
                stats.nearPercentile;

            const currentIndex =
                count > 0
                    ? 100 * (
                        currentWeightSpace * spaceComponent
                        +
                        (1 - currentWeightSpace) * nearComponent
                    )
                    : null;

            total += count;

            if (count > 0) {
                matchingZipcodes += 1;

                if (stats.labelPosition) {
                    labels.push({
                        position: [
                            stats.labelPosition[0],
                            stats.labelPosition[1],
                            count * C.heightPerHouse + 2
                        ],
                        text: String(count)
                    });
                }
            }

            return {
                ...feature,

                properties: {
                    zipcode,
                    count,

                    index: currentIndex,

                    medianLiving:
                        selection.medianLiving,

                    medianPricePerQm:
                        selection.medianPricePerQm,

                    medianKm:
                        stats.medianKm,

                    zipPricePerQm:
                        stats.zipPricePerQm,

                    spaceComponent:
                        spaceComponent === null
                            ? null
                            : spaceComponent * 100,

                    nearComponent:
                        nearComponent * 100,

                    weightSpace:
                        currentWeightSpace * 100,

                    weightNear:
                        (1 - currentWeightSpace) * 100,

                    lowSample:
                        count > 0 &&
                        count < C.minHousesForRanking,

                    minBeds: currentMinBeds,

                    budgetLow: range.low,
                    budgetHigh: range.high,

                    color:
                        count > 0
                            ? indexColor(currentIndex)
                            : C.noMatch
                }
            };
        });

    document.getElementById("summary").textContent =
        fmt(total) +
        " Häuser mit " + currentMinBeds +
        "+ Schlafzimmern in " +
        fmt(matchingZipcodes) +
        " ZIP-Gebieten";

    // Klare Empfehlung: Top-N nach Index, nur mit genug Auswahl
    const ranking = features
        .map(feature => feature.properties)
        .filter(p => p.count >= C.minHousesForRanking)
        .sort((a, b) => b.index - a.index)
        .slice(0, C.topN);

    document.getElementById("topList").innerHTML =
        ranking.length === 0
            ? "<li>Keine Treffer – Spanne anpassen.</li>"
            : ranking.map(p =>
                "<li><span class='swatch' style='background:" +
                rgb(p.color) + "'></span><b>" +
                p.zipcode + "</b> – Index " +
                fmt(p.index) + "<br>" +
                fmt(p.count) + " Häuser · " +
                fmt(p.medianLiving) + " m² · " +
                fmt(p.medianKm, 1) + " km" +
                "</li>"
            ).join("");

    return [
        // ZIP-Flächen
        new deck.GeoJsonLayer({
            id: "zip-areas",

            data: {
                type: "FeatureCollection",
                features
            },

            pickable: true,
            autoHighlight: true,

            filled: true,
            stroked: true,
            extruded: true,
            wireframe: false,

            getElevation: feature =>
                feature.properties.count *
                C.heightPerHouse,

            getFillColor: feature =>
                feature.properties.color,

            getLineColor: [255, 255, 255],

            lineWidthMinPixels: 1,
            opacity: 0.90
        }),

        // Seattle-Stadtgrenze
        new deck.PathLayer({
            id: "seattle-boundary",

            data: boundaryPaths,

            visible: showBoundaryBox.checked,

            pickable: false,

            getPath: item =>
                item.path,

            getColor:
                C.boundaryColor,

            getWidth:
                C.boundaryWidth,

            widthUnits: "pixels",
            widthMinPixels:
                C.boundaryWidth,

            jointRounded: true,
            capRounded: true,

            opacity: 1,

            parameters: {
                depthTest: false
            }
        }),

        // Downtown als Bezugspunkt der Entfernung
        new deck.ScatterplotLayer({
            id: "downtown",

            data: [{position: C.downtown}],

            visible: showDowntownBox.checked,

            pickable: false,

            getPosition: item =>
                item.position,

            getFillColor:
                C.downtownColor,

            getLineColor: [255, 255, 255],

            stroked: true,
            lineWidthMinPixels: 2,

            radiusUnits: "pixels",
            getRadius: 7,

            parameters: {
                depthTest: false
            }
        }),

        // Zahlen ohne Hintergrundkasten
        new deck.TextLayer({
            id: "house-count-labels",

            data: labels,

            pickable: false,

            getPosition: item =>
                item.position,

            getText: item =>
                item.text,

            getSize:
                C.labelSize,

            sizeUnits: "pixels",

            getColor:
                C.labelColor,

            fontFamily: "Arial, sans-serif",
            fontWeight: "bold",

            getTextAnchor: "middle",
            getAlignmentBaseline: "center",

            billboard:
                C.labelBillboard,

            background: false,

            parameters: {
                depthTest: true
            }
        })
    ];
}


// ------------------------------------------------------------
// Karte nur einmal erzeugen
// ------------------------------------------------------------

const map = new deck.DeckGL({
    container: "map",

    mapLib: maplibregl,

    mapStyle:
        "https://basemaps.cartocdn.com/gl/" +
        "positron-gl-style/style.json",

    initialViewState: C.view,

    controller: true,

    layers: buildLayers(
        currentRange()
    ),

    getTooltip: ({object, layer}) => {
        if (
            !object ||
            layer.id !== "zip-areas"
        ) {
            return null;
        }

        const p =
            object.properties;

        const usd = value =>
            fmt(value) + " $";

        let html =
            "<b>Postleitzahl: " +
            p.zipcode +
            "</b><br><br>" +

            "Preisspanne: " +
            fmtUnit(p.budgetLow) +
            " – " +
            fmtHigh(p.budgetHigh) +
            " (× 100.000 $)<br>" +

            "Mindestens " + p.minBeds + " Schlafzimmer<br>" +

            "Passende Häuser: <b>" +
            fmt(p.count) +
            "</b><br>" +

            "Entfernung zu Downtown (Median): <b>" +
            fmt(p.medianKm, 1) +
            " km</b><br><br>";

        if (p.count > 0) {
            html +=
                "Median-Wohnfläche in der Spanne: <b>" +
                fmt(p.medianLiving) +
                " m²</b><br>" +

                "Median-Preis pro m² in der Spanne: " +
                usd(p.medianPricePerQm) +
                "<br><br>" +

                "<b>Index: " +
                fmt(p.index, 1) +
                " / 100</b><br>" +

                "Gewichtung: " +
                fmt(p.weightSpace) +
                " % Wohnfläche / " +
                fmt(p.weightNear) +
                " % Nähe<br>" +

                spaceName + "-Komponente: " +
                fmt(p.spaceComponent, 1) +
                " / 100<br>" +

                "Nähekomponente: " +
                fmt(p.nearComponent, 1) +
                " / 100<br><br>";
        } else {
            html +=
                "Keine Treffer in dieser Spanne.<br><br>";
        }

        html +=
            "Preis pro m² großer Häuser (ganzes ZIP, H2): " +
            usd(p.zipPricePerQm) +

            (
                p.lowSample
                    ? "<br><br>Hinweis: wenige Treffer – " +
                      "Median unsicher, " +
                      "nicht in der Empfehlung."
                    : ""
            ) +

            "<br><br>Historische Verkäufe, " +
            "keine aktuellen Angebote.";

        return {
            html,

            style: {
                backgroundColor: "#fff",
                color: "#222",
                fontSize: "12px",
                maxWidth: "380px"
            }
        };
    },

    onError: showError
});


// ------------------------------------------------------------
// Gemeinsame Aktualisierung für alle Regler
// Kein neues IFrame, keine neue Karte, kein ViewState-Reset
// ------------------------------------------------------------

let pendingFrame = null;

function scheduleMapUpdate() {
    if (pendingFrame !== null) {
        cancelAnimationFrame(
            pendingFrame
        );
    }

    pendingFrame = requestAnimationFrame(() => {
        map.setProps({
            layers: buildLayers(currentRange())
        });

        pendingFrame = null;
    });
}


// Preisspanne ändern
// Mindestens ein Schritt Abstand zwischen von und bis.
lowSlider.addEventListener("input", () => {
    const maxLow =
        Number(highSlider.value) - C.budgetStep;

    if (Number(lowSlider.value) > maxLow) {
        lowSlider.value = maxLow;
    }

    lowSlider.style.zIndex = 3;
    highSlider.style.zIndex = 2;

    updateBudgetLabel();
    scheduleMapUpdate();
});

highSlider.addEventListener("input", () => {
    const minHigh =
        Number(lowSlider.value) + C.budgetStep;

    if (Number(highSlider.value) < minHigh) {
        highSlider.value = minHigh;
    }

    highSlider.style.zIndex = 3;
    lowSlider.style.zIndex = 2;

    updateBudgetLabel();
    scheduleMapUpdate();
});


// Stadtgrenze und Downtown ein-/ausblenden
[showBoundaryBox, showDowntownBox].forEach(box => {
    box.addEventListener("change", scheduleMapUpdate);
});


// Gewichtung ändern
weightSlider.addEventListener("input", () => {
    currentWeightSpace =
        Number(weightSlider.value) / 100;

    updateWeightLabel();
    scheduleMapUpdate();
});


// ------------------------------------------------------------
// Ansicht nur per Knopfdruck zurücksetzen
// ------------------------------------------------------------

document.getElementById(
    "resetView"
).addEventListener("click", () => {
    map.setProps({
        initialViewState: {
            ...C.view
        }
    });
});
</script>

</body>
</html>
"""


# ============================================================
# 14. HTML speichern
# ============================================================

payload_range_json = json.dumps(
    payload_range,
    ensure_ascii=False,
    allow_nan=False,
).replace("</", "<\\/")

html_range_path = Path(
    OUTPUT_HTML_RANGE
).resolve()

html_range_path.write_text(
    html_template_range.replace(
        "__PAYLOAD__",
        payload_range_json,
    ),
    encoding="utf-8",
)


# ============================================================
# 15. Lokalen Server starten (für die Anzeige in VS Code)
# ============================================================

class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass


_range_server_v3 = ThreadingHTTPServer(
    ("127.0.0.1", 0),
    partial(
        QuietHandler,
        directory=str(html_range_path.parent),
    ),
)

Thread(
    target=_range_server_v3.serve_forever,
    daemon=True,
).start()

range_url = (
    f"http://127.0.0.1:{_range_server_v3.server_address[1]}/"
    f"{quote(html_range_path.name)}"
)


# ============================================================
# 16. In Chrome öffnen
# Die HTML-Datei ist eigenständig (Daten eingebettet), daher
# als Datei öffnen: funktioniert auch nach Kernel-Neustart.
# ============================================================

def open_in_chrome(path):
    file_url = path.as_uri()

    try:
        if sys.platform == "darwin":
            subprocess.run(
                ["open", "-a", "Google Chrome", file_url],
                check=True,
            )
            return True

        for name in ["chrome", "google-chrome", "chromium"]:
            try:
                webbrowser.get(name).open(file_url)
                return True
            except webbrowser.Error:
                continue
    except Exception as error:
        print(f"Chrome konnte nicht geöffnet werden: {error}")

    # Fallback: Standardbrowser
    webbrowser.open(file_url)
    return False


if OPEN_IN_CHROME:
    open_in_chrome(html_range_path)

print(f"Karte gespeichert: {html_range_path}")


# ============================================================
# 17. Zusätzlich direkt in VS Code anzeigen
# ============================================================

display(
    IFrame(
        src=range_url,
        width="100%",
        height=950,
    )
)
